In [86]:
import boto3
import pandas as pd
import streamlit as st
import awswrangler as wr
import plotly.express as px
from datetime import datetime

In [30]:
session = boto3.Session(profile_name='hiv-project')
s3 = session.client('s3')
bucket_name = 'hiv-data-022784797781'

In [31]:
df_tables = wr.athena.read_sql_query(
    sql="select table_name from information_schema.tables where table_schema = 'hivdb_tce'",
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

df_tables

,table_name
0,dashboard
1,dim_regimen
2,dim_relative_time
3,dim_time
4,fact_measurements
5,fact_tce
6,fact_tce_old
7,mart_past_regimens
8,mart_yearly_summary
9,stg_measurements


In [32]:
sql = """
select
    regimen_signature,
    avg(baseline_log_rna) as avg_baseline_log_rna
from hivdb_tce.dashboard
where regimen_signature is not null
  and baseline_log_rna is not null
group by regimen_signature
order by avg_baseline_log_rna desc
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.bar(
    df,
    x="avg_baseline_log_rna",
    y="regimen_signature",
    orientation="h"
)

fig.update_xaxes(
    range=[df["avg_baseline_log_rna"].min() - 0.1,
           df["avg_baseline_log_rna"].max() + 0.1]
)

fig.show()
fig.write_image("regimen_vs_rna.png")

In [37]:
sql = """
select
    regimen_signature,
    avg(baseline_cd4) as avg_baseline_cd4
from hivdb_tce.dashboard
where regimen_signature is not null
  and baseline_cd4 is not null
group by regimen_signature
order by avg_baseline_cd4 desc
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.bar(
    df,
    x="avg_baseline_cd4",
    y="regimen_signature",
    orientation="h"
)

pad = (df["avg_baseline_cd4"].max() - df["avg_baseline_cd4"].min()) * 0.05

fig.update_xaxes(
    range=[
        df["avg_baseline_cd4"].min() - pad,
        df["avg_baseline_cd4"].max() + pad
    ]
)

fig.show()
fig.write_image("regimen_vs_cd4.png")

In [40]:
sql = """
select
    baseline_year,
    avg(baseline_cd4) as avg_baseline_cd4
from hivdb_tce.dashboard_unfiltered
where baseline_year is not null
  and baseline_cd4 is not null
group by baseline_year
order by baseline_year
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.line(
    df,
    x="baseline_year",
    y="avg_baseline_cd4",
    markers=True
)

fig.show()
fig.write_image("baseline_cd4_vs_baseline_year_line.png")

In [61]:
sql = """
select
    year_bucket,
    count(*) as tce_count
from hivdb_tce.dashboard_unfiltered
where year_bucket is not null
group by year_bucket
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

fig = px.pie(
    df,
    names="year_bucket",
    values="tce_count",
    hole=0.4
)

fig.show()
fig.write_image("year_bucket_breakdown.png")

"Of all TCEs in the dataset, what percentage had at least one drug of this class in their new regimen

In [48]:
sql = """
select distinct drug_class
from hivdb_tce.tce_treatments
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

print(df)

  drug_class
0       NRTI
1        CRI
2         FI
3           
4      NNRTI
5         PI
6        INI


How many unique combinations of drug classes exist in the dataset?

In [68]:
sql = """
select
    class_signature,
    count(*) as tce_count
from (
    select
        xml_filename,
        treatment_type,
        array_join(array_sort(array_agg(distinct drug_class)), '+') as class_signature
    from hivdb_tce.tce_treatments
    where drug_code is not null
      and drug_class is not null
      and trim(drug_class) != ''
    group by xml_filename, treatment_type
)
group by class_signature
order by tce_count desc
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

print(df)

          class_signature  tce_count
0                 NRTI+PI       1280
1           NNRTI+NRTI+PI       1026
2              NNRTI+NRTI        398
3                    NRTI        154
4             INI+NRTI+PI         44
5              FI+NRTI+PI         28
6        FI+NNRTI+NRTI+PI         23
7                NNRTI+PI         21
8       INI+NNRTI+NRTI+PI         18
9                INI+NRTI          7
10         FI+INI+NRTI+PI          6
11             CRI+INI+PI          5
12                 INI+PI          5
13         INI+NNRTI+NRTI          4
14        CRI+INI+NRTI+PI          4
15           CRI+INI+NRTI          4
16           INI+NNRTI+PI          3
17            CRI+NRTI+PI          3
18                     PI          3
19         CRI+FI+NRTI+PI          2
20   FI+INI+NNRTI+NRTI+PI          2
21          CRI+INI+NNRTI          2
22      FI+INI+NNRTI+NRTI          2
23              INI+NNRTI          1
24  CRI+INI+NNRTI+NRTI+PI          1
25     CRI+INI+NNRTI+NRTI          1
2

In [69]:
threshold = 20

df_plot = df.copy()
df_plot.loc[df_plot['tce_count'] < threshold, 'class_signature'] = 'Other'
df_plot = df_plot.groupby('class_signature', as_index=False)['tce_count'].sum()

df_plot = df_plot.sort_values('tce_count', ascending=False)
other_row = df_plot[df_plot['class_signature'] == 'Other']
df_plot = pd.concat([df_plot[df_plot['class_signature'] != 'Other'], other_row])

color_map = {sig: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, sig in enumerate(df_plot[df_plot['class_signature'] != 'Other']['class_signature'])}
color_map['Other'] = '#808080'

legend_order = list(df_plot[df_plot['class_signature'] != 'Other']['class_signature']) + ['Other']

fig = px.pie(
    df_plot,
    names="class_signature",
    values="tce_count",
    hole=0.4,
    color="class_signature",
    color_discrete_map=color_map,
    category_orders={"class_signature": legend_order}
)

fig.show()
fig.write_image("drug_class_combinations.png")

In [72]:
sql = """
select
    class_signature,
    count(*) as tce_count
from (
    select
        xml_filename,
        treatment_type,
        array_join(array_sort(array_agg(distinct drug_class)), '+') as class_signature
    from hivdb_tce.tce_treatments
    where drug_code is not null
      and drug_class is not null
      and trim(drug_class) != ''
      and treatment_type = 'new'
    group by xml_filename, treatment_type
)
group by class_signature
order by tce_count desc
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

print(df)

          class_signature  tce_count
0                 NRTI+PI        741
1           NNRTI+NRTI+PI        317
2              NNRTI+NRTI        231
3                    NRTI         70
4             INI+NRTI+PI         44
5              FI+NRTI+PI         25
6                NNRTI+PI         19
7       INI+NNRTI+NRTI+PI         13
8        FI+NNRTI+NRTI+PI          9
9                INI+NRTI          7
10         FI+INI+NRTI+PI          6
11                 INI+PI          5
12             CRI+INI+PI          5
13        CRI+INI+NRTI+PI          4
14           CRI+INI+NRTI          4
15         INI+NNRTI+NRTI          4
16           INI+NNRTI+PI          3
17                     PI          3
18          CRI+INI+NNRTI          2
19            CRI+NRTI+PI          2
20         CRI+FI+NRTI+PI          2
21      FI+INI+NNRTI+NRTI          2
22          FI+NNRTI+NRTI          1
23  CRI+INI+NNRTI+NRTI+PI          1
24            FI+NNRTI+PI          1
25              FI+INI+PI          1
2

#### Which combination of drug classes were patients switched to after TCE?

In [79]:
threshold = 7

df_plot = df.copy()
df_plot.loc[df_plot['tce_count'] < threshold, 'class_signature'] = 'Other'
df_plot = df_plot.groupby('class_signature', as_index=False)['tce_count'].sum()

df_plot = df_plot.sort_values('tce_count', ascending=False)
other_row = df_plot[df_plot['class_signature'] == 'Other']
df_plot = pd.concat([df_plot[df_plot['class_signature'] != 'Other'], other_row])

colors = px.colors.qualitative.Prism
color_map = {sig: colors[i % len(colors)]
             for i, sig in enumerate(df_plot[df_plot['class_signature'] != 'Other']['class_signature'])}
color_map['Other'] = '#808080'

legend_order = list(df_plot[df_plot['class_signature'] != 'Other']['class_signature']) + ['Other']

fig = px.pie(
    df_plot,
    names="class_signature",
    values="tce_count",
    hole=0.4,
    color="class_signature",
    color_discrete_map=color_map,
    category_orders={"class_signature": legend_order}
)

fig.show()
fig.write_image("drug_class_combinations.png")

In [80]:
sql = """
SELECT * 
FROM hivdb_tce.v_new_regimen_class_counts
ORDER BY tce_count DESC;
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

print(df)

          class_signature  tce_count
0                 NRTI+PI        741
1           NNRTI+NRTI+PI        317
2              NNRTI+NRTI        231
3                    NRTI         70
4             INI+NRTI+PI         44
5              FI+NRTI+PI         25
6                NNRTI+PI         19
7       INI+NNRTI+NRTI+PI         13
8        FI+NNRTI+NRTI+PI          9
9                INI+NRTI          7
10         FI+INI+NRTI+PI          6
11             CRI+INI+PI          5
12                 INI+PI          5
13         INI+NNRTI+NRTI          4
14        CRI+INI+NRTI+PI          4
15           CRI+INI+NRTI          4
16           INI+NNRTI+PI          3
17                     PI          3
18            CRI+NRTI+PI          2
19         CRI+FI+NRTI+PI          2
20      FI+INI+NNRTI+NRTI          2
21          CRI+INI+NNRTI          2
22                  FI+PI          1
23     CRI+INI+NNRTI+NRTI          1
24            FI+NNRTI+PI          1
25          FI+NNRTI+NRTI          1
2

In [81]:
threshold = 7

df_plot = df.copy()
df_plot.loc[df_plot['tce_count'] < threshold, 'class_signature'] = 'Other'
df_plot = df_plot.groupby('class_signature', as_index=False)['tce_count'].sum()

df_plot = df_plot.sort_values('tce_count', ascending=False)
other_row = df_plot[df_plot['class_signature'] == 'Other']
df_plot = pd.concat([df_plot[df_plot['class_signature'] != 'Other'], other_row])

colors = px.colors.qualitative.Prism
color_map = {sig: colors[i % len(colors)]
             for i, sig in enumerate(df_plot[df_plot['class_signature'] != 'Other']['class_signature'])}
color_map['Other'] = '#808080'

legend_order = list(df_plot[df_plot['class_signature'] != 'Other']['class_signature']) + ['Other']

fig = px.pie(
    df_plot,
    names="class_signature",
    values="tce_count",
    hole=0.4,
    color="class_signature",
    color_discrete_map=color_map,
    category_orders={"class_signature": legend_order}
)

fig.show()
fig.write_image("drug_class_combinations.png")

In [82]:
sql = """
select
    t.xml_filename,
    t.drug_code,
    t.drug_class
from hivdb_tce.tce_treatments t
where t.treatment_type = 'new'
  and t.xml_filename in (
      select xml_filename
      from hivdb_tce.tce_treatments
      where treatment_type = 'new'
      group by xml_filename
      having array_join(array_sort(array_agg(distinct drug_class)), '+') = 'NRTI'
  )
order by t.xml_filename;
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

print(df)

    xml_filename drug_code drug_class
0       1010.xml       3TC       NRTI
1       1010.xml       AZT       NRTI
2       1010.xml       ABC       NRTI
3       1010.xml       DDI       NRTI
4       1011.xml       AZT       NRTI
..           ...       ...        ...
215      974.xml       DDI       NRTI
216      974.xml       TDF       NRTI
217       98.xml       3TC       NRTI
218       98.xml       AZT       NRTI
219       98.xml       ABC       NRTI

[220 rows x 3 columns]


#### TCE Frequency by Year

In [90]:
sql = """
select * from hivdb_tce.tce_by_year order by baseline_year
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

df["baseline_year"] = df["baseline_year"].astype(int)
df = df.sort_values("baseline_year")

fig = px.line(
    df,
    x="baseline_year",
    y="tce_count",
    markers=True
)

fig.update_layout(
    title="TCE Frequency by Year",
    xaxis_title="Baseline Year",
    yaxis_title="Number of Treatment Change Events"
)

fig.show()